# Working with Fourier Transforms

This tutorial explores Fourier transforms in HCIPy, from basic usage to advanced techniques. We'll see how different Fourier transform algorithms work, when to use each one, and how to perform operations in the Fourier domain.



## Why Fourier Transforms in Optics?

Fourier transforms are fundamental to optical simulations. They allow us to:

* **Propagate light** between pupil and focal planes (Fraunhofer diffraction)
* **Analyze spatial frequencies** in wavefronts and images
* **Filter images** by manipulating the frequency domain
* **Calculate PSFs and MTFs** for optical system characterization
* **Perform image registration** via shifts, rotations, and distortions

HCIPy provides multiple Fourier transform implementations optimized for different use cases.

In [ ]:
from hcipy import *
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

## Understanding FFT Grids

When we compute a discrete Fourier transform, the relationship between the spatial domain grid and the Fourier domain grid is determined by the sampling theorem. Let's explore this relationship.

In [ ]:
# Create a spatial grid
N = 128
spatial_grid = make_pupil_grid(N, 2.0)  # 128 points spanning [-1, 1]

print("Spatial grid properties:")
print(f"  Dimensions: {spatial_grid.dims}")
print(f"  Spacing: {spatial_grid.delta}")
print(f"  Total extent: {spatial_grid.delta[0] * spatial_grid.dims[0]:.4f}")

The Fourier domain grid has a reciprocal relationship with the spatial grid. The maximum frequency (Nyquist frequency) is determined by the spatial sampling, and the frequency resolution is determined by the total spatial extent.

In [ ]:
# Create the corresponding FFT grid
fft_grid = make_fft_grid(spatial_grid)

print("\nFFT grid properties:")
print(f"  Dimensions: {fft_grid.dims}")
print(f"  Spacing: {fft_grid.delta}")
print(f"  Zero point: {fft_grid.zero}")

# Calculate the relationship
dx = spatial_grid.delta[0]
N = spatial_grid.dims[0]
L = dx * N
df = 2 * np.pi / L  # Frequency spacing

print(f"\nTheoretical relationships:")
print(f"  Spatial extent L = {L:.4f}")
print(f"  Frequency spacing df = 2π/L = {df:.4f}")
print(f"  Maximum frequency = π/dx = {np.pi/dx:.4f}")

Let's visualize the relationship between spatial and frequency grids.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot spatial grid points (1D slice for visualization)
# Take a 1D slice through the middle of the 2D grid
spatial_1d = spatial_grid.x.reshape(spatial_grid.dims)[N//2, :]
axes[0].plot(spatial_1d, np.zeros(N), 'o', markersize=2, label='Grid points')
axes[0].axvline(x=0, color='red', linestyle='--', alpha=0.5, label='Center')
axes[0].set_xlabel('x')
axes[0].set_ylabel('(zero)')
axes[0].set_title('Spatial Domain Grid (1D slice)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot frequency grid points (1D slice)
fft_1d = fft_grid.x.reshape(fft_grid.dims)[N//2, :]
axes[1].plot(fft_1d, np.zeros(N), 'o', markersize=2, label='Grid points')
axes[1].axvline(x=0, color='red', linestyle='--', alpha=0.5, label='Center')
axes[1].set_xlabel('Spatial frequency (k)')
axes[1].set_ylabel('(zero)')
axes[1].set_title('Frequency Domain Grid (1D slice)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Controlling Resolution with q and fov

In optical simulations, we often want to control the sampling in the Fourier domain independently from the spatial domain. HCIPy provides two parameters for this:

- **q (oversampling)**: Increases the number of samples in the Fourier domain
- **fov (field of view)**: Controls the extent of the Fourier domain

In [ ]:
# Create an aperture
aperture = make_circular_aperture(0.5)(spatial_grid)

# Different FFT grid configurations
configs = [
    (1, 1, 'Critical sampling'),
    (2, 1, '2x oversampling'),
    (1, 0.5, 'Half FOV'),
    (2, 0.5, '2x oversampling, half FOV')
]

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for idx, (q, fov, title) in enumerate(configs):
    # Create FFT grid with these parameters
    fft_grid_config = make_fft_grid(spatial_grid, q=q, fov=fov)

    # Create FFT object
    fft = FastFourierTransform(spatial_grid, q=q, fov=fov)

    # Compute FFT
    wf = Wavefront(aperture)
    focal_field = fft.forward(wf.electric_field)

    # Plot
    im = imshow_field(np.log10(np.abs(focal_field)**2 / (np.abs(focal_field)**2).max()), vmin=-10, grid=fft_grid_config, ax=axes[idx])
    axes[idx].set_title(f'{title}\nq={q}, fov={fov}\nGrid: {fft_grid_config.dims}')
    plt.colorbar(im, ax=axes[idx])

plt.tight_layout()
plt.show()

print("Key observations:")
print("- Higher q gives more samples (better resolution in Fourier domain)")
print("- Lower fov crops the Fourier domain (faster but may lose information)")

## Choosing the Right Transform Method

HCIPy implements several Fourier transform algorithms, each with different trade-offs:

1. **FastFourierTransform (FFT)**: Fast but requires specific grid relationships
2. **MatrixFourierTransform (MFT)**: Flexible but O(N²) complexity
3. **NaiveFourierTransform**: Direct implementation, mainly for testing
4. **ZoomFastFourierTransform**: Efficient for zoomed regions

Let's compare them.

In [ ]:
from hcipy.fourier import FastFourierTransform, MatrixFourierTransform
import time

# Create test field
grid_256 = make_pupil_grid(256)
aperture_256 = make_circular_aperture(1)(grid_256)
wf_256 = Wavefront(aperture_256)

# Test 1: FFT (native grid)
fft_grid_native = make_fft_grid(grid_256)
fft = FastFourierTransform(grid_256)

start = time.time()
for _ in range(10):
    result_fft = fft.forward(wf_256.electric_field)
fft_time = (time.time() - start) / 10

# Test 2: MFT (arbitrary output grid)
# Create a custom output grid with different sampling
custom_output_grid = make_pupil_grid(300, 15.0)  # Different size and extent
mft = MatrixFourierTransform(grid_256, custom_output_grid)

start = time.time()
for _ in range(10):
    result_mft = mft.forward(wf_256.electric_field)
mft_time = (time.time() - start) / 10

print(f"FFT time: {fft_time*1000:.2f} ms")
print(f"MFT time: {mft_time*1000:.2f} ms")
print(f"Speedup: {mft_time/fft_time:.1f}x")

# Visualize both results
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

imshow_field(np.log10(np.abs(result_fft)**2 / (np.abs(result_fft)**2).max()),
             vmin=-10, grid=fft_grid_native, ax=axes[0])
axes[0].set_title(f'FFT (native grid)\n{fft_grid_native.dims} pixels')

imshow_field(np.log10(np.abs(result_mft)**2 / (np.abs(result_mft)**2).max()),
             vmin=-10, grid=custom_output_grid, ax=axes[1])
axes[1].set_title(f'MFT (custom grid)\n{custom_output_grid.dims} pixels')

plt.tight_layout()
plt.show()

## Filtering in the Fourier Domain

Fourier filtering is a powerful technique for modifying fields in the frequency domain. HCIPy provides `FourierFilter` for efficient filtering operations.

In [ ]:
from hcipy.fourier import FourierFilter

# Create a field with noise
grid_128 = make_pupil_grid(128)
x = grid_128.x
y = grid_128.y

# Create a pattern with high-frequency noise
signal = np.exp(-(x**2 + y**2) / 0.1)  # Gaussian
noise = 0.1 * np.sin(2 * np.pi * 10 * x) * np.sin(2 * np.pi * 10 * y)
noisy_field = Field(signal + noise, grid_128)

# Create a low-pass filter in Fourier domain
def low_pass_filter(grid):
    k = np.sqrt(grid.x**2 + grid.y**2)
    cutoff = 20.0
    return Field(np.exp(-(k / cutoff)**2), grid)

# Apply filter
filt = FourierFilter(grid_128, low_pass_filter, q=2)
filtered_field = filt.forward(noisy_field)

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

imshow_field(noisy_field, ax=axes[0])
axes[0].set_title('Noisy Input')

# Get the internal FFT grid to show the filter
fft_grid_internal = filt.internal_grid
filter_field = low_pass_filter(fft_grid_internal)
imshow_field(filter_field, ax=axes[1])
axes[1].set_title('Low-Pass Filter\n(Fourier Domain)')

imshow_field(filtered_field.real, ax=axes[2])
axes[2].set_title('Filtered Output')

plt.tight_layout()
plt.show()

## Image Transformations

HCIPy provides several operators that work in the Fourier domain: shifts, shears, and rotations. These are particularly useful for image registration and distortion correction.

In [ ]:
# Create a test pattern - the letter F
grid_2d = make_pupil_grid(128, 2.0)

def f_pattern(grid):
    # Create an F pattern using multiple rectangular regions
    pattern = grid.zeros()

    vertical_bar = make_rectangular_aperture((0.15, 1.15), center=(-0.3, 0))
    top_bar = make_rectangular_aperture((0.5, 0.15), center=(0, 0.5))
    middle_bar = make_rectangular_aperture((0.5, 0.15), center=(0, 0))

    pattern[vertical_bar(grid) > 0.5] = 1
    pattern[top_bar(grid) > 0.5] = 1
    pattern[middle_bar(grid) > 0.5] = 1

    return pattern

test_pattern = f_pattern(grid_2d)

# Apply operations
shifter = FourierShift(grid_2d, [0.2, 0.3])
shifted = shifter.forward(test_pattern)

shearer = FourierShear(grid_2d, shear=0.3, shear_dim=0)
sheared = shearer.forward(test_pattern)

rotator = FourierRotation(grid_2d, angle=np.pi/6)
rotated = rotator.forward(test_pattern)

# Visualize
fig, axes = plt.subplots(2, 2, figsize=(12, 12))

imshow_field(test_pattern, ax=axes[0,0], cmap='gray')
axes[0,0].set_title('Original F Pattern')

imshow_field(shifted.real, ax=axes[0,1], cmap='gray')
axes[0,1].set_title('Shifted by (0.2, 0.3)')

imshow_field(sheared.real, ax=axes[1,0], cmap='gray')
axes[1,0].set_title('Sheared (x-direction)')

imshow_field(rotated.real, ax=axes[1,1], cmap='gray')
axes[1,1].set_title('Rotated by 30°')

plt.tight_layout()
plt.show()

## Calculating the MTF

As a practical example, let's calculate the Modulation Transfer Function (MTF) of an optical system with a square aperture.

In [ ]:
# Create optical system
pupil_grid = make_pupil_grid(256, 2.0)

# Square aperture
aperture = make_rectangular_aperture([1.0, 1.0])(pupil_grid)

# Create FFT for PSF calculation
focal_grid = make_fft_grid(pupil_grid, q=2)
fft = FastFourierTransform(pupil_grid, q=2)

# Calculate PSF
wf = Wavefront(aperture)
psf = np.abs(fft.forward(wf.electric_field))**2
psf /= psf.max()

# Calculate MTF (autocorrelation of pupil = |FFT{PSF}|)
# The MTF is the normalized magnitude of the Fourier transform of the PSF
mtf_grid = make_fft_grid(focal_grid)
fft_for_mtf = FastFourierTransform(focal_grid)
mtf_complex = fft_for_mtf.forward(psf)
mtf = np.abs(mtf_complex) / np.abs(mtf_complex).max()

# Plot results
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Pupil
imshow_field(aperture, ax=axes[0], cmap='gray')
axes[0].set_title('Square Aperture (Pupil)')

# PSF
imshow_field(np.log10(psf), vmin=-5, grid=focal_grid, ax=axes[1])
axes[1].set_title('Point Spread Function (log scale)')

# MTF
imshow_field(mtf, grid=mtf_grid, ax=axes[2])
axes[2].set_title('Modulation Transfer Function')

plt.tight_layout()
plt.show()

print("The MTF shows how contrast is transferred at different spatial frequencies.")
print("For a square aperture, the MTF has a triangular shape (sinc² pattern).")

## Performance Considerations

When performing many Fourier transforms, it's important to choose the right algorithm. HCIPy can automatically select the best method using `make_fourier_transform()`.

In [ ]:
from hcipy.fourier import make_fourier_transform
import time

# Use a large grid to demonstrate performance differences
grid_large = make_pupil_grid(1024)
field = grid_large.zeros(dtype='complex')

# Example 1: Full field of view - FFT is optimal
fft_transform = make_fourier_transform(grid_large, fov=1)
print(f"Full FOV (fov=1): {type(fft_transform).__name__}")
print(f"  Output grid: {fft_transform.output_grid.dims}")

# Example 2: Small zoomed region - MFT is optimal
# When we only need a small region with high resolution, MFT is faster
mft_transform = make_fourier_transform(grid_large, fov=1/32, q=4)
print(f"\nZoomed FOV (fov=1/32, q=4): {type(mft_transform).__name__}")
print(f"  Output grid: {mft_transform.output_grid.dims}")


## Summary

In this tutorial, we explored:

1. **FFT Grid Relationships**: Understanding how spatial and frequency grids relate
2. **Zero-Padding and FOV**: Controlling Fourier domain sampling with `q` and `fov` parameters
3. **Fourier Transform Methods**: Comparing FFT (fast, specific grids) vs MFT (flexible, slower)
4. **Fourier Filtering**: Using `FourierFilter` for frequency-domain operations
5. **Fourier Operations**: Shifts, shears, and rotations in the Fourier domain
6. **Practical Applications**: Calculating MTF from PSF
7. **Performance**: FFT scales as O(N log N) while MFT scales as O(N²)

For most optical simulations, use `FastFourierTransform` with appropriate `q` and `fov` parameters. Use `MatrixFourierTransform` only when you need arbitrary output grids.